In [1]:
#导入必要库
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft2, ifft2, fftshift, ifftshift
import scipy.ndimage
plt.rcParams['font.sans-serif'] = ['SimHei']  # 设置字体为黑体
plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题

In [2]:
# 单位定义
cm = 1e-2
mm = 1e-3
um = 1e-6
nm = 1e-9

In [3]:
# 传递函数法计算衍射图像
def propTF(u1, L, lambda_, z):

    M, N = u1.shape
    dx = L / M  # 采样间隔
    k = 2 * np.pi / lambda_  # 波数

    # 频率坐标
    fx = np.linspace(-1 / (2 * dx), 1 / (2 * dx) - 1 / L, N)
    FX, FY = np.meshgrid(fx, fx)

    # 传递函数
    H = np.exp(-1j * np.pi * lambda_ * z * (FX ** 2 + FY ** 2))
    H = fftshift(H)  # 频谱中心化

    # FFT变换
    U1 = fft2(fftshift(u1))
    U2 = H * U1
    u2 = ifftshift(ifft2(U2))
    return u2


In [4]:
# 薄散射层模拟函数
def ThinDiffuser(e, w, N):

    x = np.linspace(-N / 2, N / 2 - 1, N)
    y = np.linspace(-N / 2, N / 2 - 1, N)
    X, Y = np.meshgrid(x, y)
    r2 = X ** 2 + Y ** 2

    # 高斯相关函数
    G = w * e * 2 * np.sqrt(3 / np.pi)
    g_kernel = G / (2 * np.pi * w ** 2) * np.exp(-0.5 * r2 / w ** 2)

    # 生成随机相位并滤波
    delta = 2 * np.pi * (np.random.rand(N, N) - 0.5)
    F_g = fft2(fftshift(g_kernel))
    F_delta = fft2(fftshift(delta))
    delta_filtered = np.real(ifftshift(ifft2(F_g * F_delta)))
    return delta_filtered


In [5]:
# 厚散射体模拟函数
def ThickDiffuser(thick, ls, g, lambda_, px, N, N_diff=10):

    k = 2 * np.pi / lambda_
    L = thick  # 使用厚度作为特征长度
    Theta_0 = np.sqrt(L * (1 - g) / ls)  # 散射角范围

    if Theta_0 > np.pi / 4:
        print('WARNING: 输出散射角大于 pi/4 - 超过模型限制')

    # 计算表面参数
    e = np.sqrt(L / (N_diff * ls * k ** 2))
    w = np.sqrt(N_diff / 2) * e / (px * Theta_0)

    # 生成多层散射体
    delta = np.zeros((N, N, N_diff))
    for j in range(N_diff):
        delta[:, :, j] = ThinDiffuser(e, w, N)

    return delta, Theta_0


In [11]:
def main():
    # 参数设置
    lambda_ = 633 * nm
    k = 2 * np.pi / lambda_

    N = 256  # 空间采样点数
    L = 3 * mm  # 模拟区域尺寸
    delta = L / N  # 采样间隔

    u = 6 * cm  # 光源到散射体的距离
    v = 6 * cm  # 散射体到观测面的距离

    # 创建坐标网格
    x = np.linspace(-N / 2, N / 2 - 1, N) * delta
    y = np.linspace(-N / 2, N / 2 - 1, N) * delta
    X, Y = np.meshgrid(x, y)

    # 高斯光束
    A = 1
    z = 0 * cm
    w0 = 0.48 * mm  # 束腰半径
    wz = w0 * np.sqrt(1 + (lambda_ * z / (np.pi * w0 ** 2)) ** 2)
    E0 = A * (w0 / wz) * np.exp(-(X ** 2 + Y ** 2) / wz ** 2)

    # 传播到散射体表面
    E1 = propTF(E0, L, lambda_, u)

    # 生成厚散射体
    ls = 100 * um  # 散射平均自由程
    thick = 500 * um  # 散射体厚度
    g = 0.98  # 各向异性因子
    N_diff = 5  # 散射层数
    d = thick / (N_diff - 1)  # 层间距离

    Diffuser, Theta_0 = ThickDiffuser(thick, ls, g, lambda_, delta, N, N_diff)
# 光通过散射体传播
    E_s_old = E1
    for n in range(N_diff):
        E_s_new = E_s_old * np.exp(1j * k * Diffuser[:, :, n])  # 随机相位散射
        E_s_old = propTF(E_s_new, L, lambda_, d)  # 层间传播
    Eout = propTF(E_s_new, L, lambda_, v)  # 传播到观测面
    S0 = np.abs(Eout) ** 2  # 光强分布(散斑图)

    # 显示初始散斑图
    plt.figure(figsize=(8, 6))
    plt.imshow(S0, cmap='hot', extent=[x.min(), x.max(), y.min(), y.max()])
    plt.title('初始位置散斑')
    plt.xticks([])  # 清除x轴刻度
    plt.yticks([])  # 清除y轴刻度

    plt.show()

    # 倾斜入射角度扫描
    deg = np.pi / 180
    angles = np.arange(-0.06, 0.062, 0.002)
    correlations = np.zeros(len(angles))

    for i, alpha in enumerate(angles):
        # 添加线性相位模拟倾斜
        E1_tilt = E0 * np.exp(1j * k * X * np.tan(alpha * deg))

        # 传播到散射体
        E2_tilt = propTF(E1_tilt, L, lambda_, u)

        # 通过散射体传播
        E_s_old_t = E2_tilt
        for n in range(N_diff):
            E_s_new_t = E_s_old_t * np.exp(1j * k * Diffuser[:, :, n])
            E_s_old_t = propTF(E_s_new_t, L, lambda_, d)

        # 传播到观测面并计算光强
        Eout_tilt = propTF(E_s_new_t, L, lambda_, v)
        S1 = np.abs(Eout_tilt) ** 2
        plt.imshow(S1,cmap='hot', extent=[x.min(), x.max(), y.min(), y.max()])
        plt.title(f'旋转角度{alpha:2f}位置散斑')
        plt.xticks([])  # 清除x轴刻度
        plt.yticks([])  # 清除y轴刻度

        plt.show()
        #计算相关系数
        correlations[i] = np.corrcoef(S0.ravel(), S1.ravel())[0, 1]

     #绘制记忆效应曲线
    plt.figure(figsize=(10, 6))
    plt.plot(angles, correlations, 'b-', linewidth=2)
    plt.axhline(y=0.5, color='g', linestyle='--', linewidth=2)
    plt.xlabel('入射光倾斜角度(deg)', fontsize=12)
    plt.ylabel('相关系数（归一化）', fontsize=12)
    plt.title('模拟的记忆效应曲线', fontsize=14)
    plt.grid(True)
    plt.legend(['相关系数', '0.5'], fontsize=10)
    plt.tight_layout()
    plt.show()


In [ ]:
if __name__ == "__main__":
    main()
